In [0]:
# ============================================================
# CAPA GOLD: Agregaciones y métricas de negocio listas para consumo
# ============================================================

print("📥 Leyendo tabla limpia silver_telemetry...")
df_silver = spark.read.table("silver_telemetry")

# 1. Extraer la fecha del timestamp para poder agrupar por día
# Usamos funciones de PySpark SQL para manipular fechas
from pyspark.sql.functions import col, to_date, avg, max, count, round

df_with_date = df_silver.withColumn("report_date", to_date(col("timestamp")))

# 2. Crear la agregación (Resumen Diario por Dispositivo)
print("⚙️ Calculando agregaciones diarias por dispositivo...")
df_gold = df_with_date.groupBy("device_id", "report_date").agg(
    count("event_id").alias("total_events"),
    round(avg("speed_kmh"), 2).alias("avg_speed_kmh"),
    max("engine_temp_c").alias("max_engine_temp_c")
)

# 3. Ordenar para verlo bonito (por fecha descendente y luego por dispositivo)
df_gold = df_gold.orderBy(col("report_date").desc(), col("device_id"))

print(f"✅ Agregación completada. Filas generadas: {df_gold.count()}")

# 4. Vista previa
print("\n📊 Vista previa de la Capa Gold (Resumen Diario):")
display(df_gold.limit(10))

# 5. Guardar como tabla Delta gestionada "gold_device_daily_summary"
print("\n💾 Guardando como tabla gold_device_daily_summary...")
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_device_daily_summary")

print("\n✅ ¡ÉXITO TOTAL! Tu arquitectura Lakehouse de 3 capas está completa:")
print("   🥉 Bronze: Datos crudos (10,000 filas)")
print("   🥈 Silver: Datos limpios y deduplicados (9,243 filas)")
print("   🥇 Gold:   Datos agregados listos para consumo")